In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += ";" + r"C:\hadoop\bin"

In [29]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

import pandas as pd
from datetime import datetime

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

import time

start_time = time.time()

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("MegaMart")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.hadoop.hadoop.home.dir",
        r"C:\hadoop"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


spark.sparkContext._jsc.hadoopConfiguration().set(
    "hadoop.home.dir",
    r"C:\hadoop"
)

run_id = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


MegaMart receives transactions from multiple retail channels.

Business Key:
- transaction_id

CDC Column:
- updated_at

Requirements:
- Incremental Load
- Watermark
- Late Arriving Data (7 Days)
- DLQ
- Merge Upsert
- Logging

In [3]:
#data dummy
hyper_pd = pd.DataFrame(
    {
        "transaction_id": [
            "TX001",
            "TX002",
            "TX003"
        ],
        "transaction_date": [
            "2026-01-01",
            "2026-01-01",
            "2026-01-02"
        ],
        "store_id": [
            "S001",
            "S001",
            "S002"
        ],
        "sales_amount": [
            100,
            200,
            300
        ],
        "updated_at": [
            "2026-01-01 10:00:00",
            "2026-01-01 11:00:00",
            "2026-01-02 09:00:00"
        ]
    }
)

In [8]:
hyper = spark.createDataFrame(
    hyper_pd
)

hyper.show()

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|
+--------------+----------------+--------+------------+-------------------+
|         TX001|      2026-01-01|    S001|         100|2026-01-01 10:00:00|
|         TX002|      2026-01-01|    S001|         200|2026-01-01 11:00:00|
|         TX003|      2026-01-02|    S002|         300|2026-01-02 09:00:00|
+--------------+----------------+--------+------------+-------------------+



Raw source data is stored in Bronze layer.

In [9]:
(
    hyper
    .write
    .format("delta")
    .mode("overwrite")
    .save(
        "../data/bronze/hyper"
    )
)

Watermark tracks the latest processed record per source.

In [10]:
watermark_pd = pd.DataFrame(
    {
        "source_name": [
            "hyper"
        ],
        "last_loaded_ts": [
            "1900-01-01 00:00:00"
        ]
    }
)

In [12]:
watermark = spark.createDataFrame(
    watermark_pd
)

(
    watermark
    .write
    .format("delta")
    .mode("overwrite")
    .save(
        "../data/metadata/watermark"
    )
)

In [5]:
watermark_df = (
    spark.read
    .format("delta")
    .load(
        "../data/metadata/watermark"
    )
)

watermark_df.show()

+-----------+-------------------+
|source_name|     last_loaded_ts|
+-----------+-------------------+
|      hyper|1900-01-01 00:00:00|
+-----------+-------------------+



In [7]:
last_watermark = (
    watermark_df
    .filter(
        F.col("source_name") == "hyper"
    )
    .select(
        "last_loaded_ts"
    )
    .collect()[0][0]
)

print(f"Last watermark for hyper: {last_watermark}")

Last watermark for hyper: 1900-01-01 00:00:00


Only records newer than watermark are processed.

In [9]:
incremental_df = (
    spark.read
    .format("delta")
    .load(
        "../data/bronze/hyper"
    )
    .filter(
        F.col("updated_at")
        > F.lit(last_watermark)
    )
)
incremental_df.show()

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|
+--------------+----------------+--------+------------+-------------------+
|         TX001|      2026-01-01|    S001|         100|2026-01-01 10:00:00|
|         TX002|      2026-01-01|    S001|         200|2026-01-01 11:00:00|
|         TX003|      2026-01-02|    S002|         300|2026-01-02 09:00:00|
+--------------+----------------+--------+------------+-------------------+



Since this is the first load, write all records to Silver.

In [11]:
(
    incremental_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(
        "../data/silver/unified_transactions_incremental"
    )
)

In [12]:
max_ts = (
    incremental_df
    .agg(
        F.max(
            "updated_at"
        )
    )
    .collect()[0][0]
)
print(f"Max updated_at in incremental_df: {max_ts}")

Max updated_at in incremental_df: 2026-01-02 09:00:00


In [ ]:
simulasi CDC

Data masuk hari berikutnya

In [14]:
hyper_run2_pd = pd.DataFrame(
    {
        "transaction_id": [
            "TX001",
            "TX004",
            "TX005",
            "TX006"
        ],
        "transaction_date": [
            "2026-01-01",
            "2026-01-03",
            "2026-01-03",
            "2025-12-29"
        ],
        "store_id": [
            "S001",
            "S003",
            "S003",
            "S001"
        ],
        "sales_amount": [
            150,
            400,
            "ERROR",
            500
        ],
        "updated_at": [
            "2026-01-03 08:00:00",
            "2026-01-03 08:05:00",
            "2026-01-03 08:10:00",
            "2026-01-03 08:15:00"
        ]
    }
)

hyper_run2 = spark.createDataFrame(
    hyper_run2_pd
)

hyper_run2.show()

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|
+--------------+----------------+--------+------------+-------------------+
|         TX001|      2026-01-01|    S001|         150|2026-01-03 08:00:00|
|         TX004|      2026-01-03|    S003|         400|2026-01-03 08:05:00|
|         TX005|      2026-01-03|    S003|       ERROR|2026-01-03 08:10:00|
|         TX006|      2025-12-29|    S001|         500|2026-01-03 08:15:00|
+--------------+----------------+--------+------------+-------------------+



Data Sebelumnya

In [18]:
incremental_df = (
    spark.read
    .format("delta")
    .load(
        "../data/bronze/hyper"
    )
    .filter(
        F.col("updated_at")
        > F.lit(last_watermark)
    )
)
incremental_df.show()

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|
+--------------+----------------+--------+------------+-------------------+
|         TX001|      2026-01-01|    S001|         100|2026-01-01 10:00:00|
|         TX002|      2026-01-01|    S001|         200|2026-01-01 11:00:00|
|         TX003|      2026-01-02|    S002|         300|2026-01-02 09:00:00|
+--------------+----------------+--------+------------+-------------------+



Handle late arriving data 7 hari

In [19]:
incremental_run2 = (
    hyper_run2
    .filter(
        F.col("updated_at")
        >= F.date_sub(
            F.lit(last_watermark),
            7
        )
    )
)

incremental_run2.show()

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|
+--------------+----------------+--------+------------+-------------------+
|         TX001|      2026-01-01|    S001|         150|2026-01-03 08:00:00|
|         TX004|      2026-01-03|    S003|         400|2026-01-03 08:05:00|
|         TX005|      2026-01-03|    S003|       ERROR|2026-01-03 08:10:00|
|         TX006|      2025-12-29|    S001|         500|2026-01-03 08:15:00|
+--------------+----------------+--------+------------+-------------------+



In [20]:
valid_df = (
    incremental_run2
    .withColumn(
        "sales_amount_num",
        F.col("sales_amount")
        .cast("double")
    )
    .filter(
        F.col("sales_amount_num")
        .isNotNull()
    )
)

invalid_df = (
    incremental_run2
    .withColumn(
        "sales_amount_num",
        F.col("sales_amount")
        .cast("double")
    )
    .filter(
        F.col("sales_amount_num")
        .isNull()
    )
)

In [23]:
invalid_df.show()

+--------------+----------------+--------+------------+-------------------+----------------+--------------------+
|transaction_id|transaction_date|store_id|sales_amount|         updated_at|sales_amount_num|        error_reason|
+--------------+----------------+--------+------------+-------------------+----------------+--------------------+
|         TX005|      2026-01-03|    S003|       ERROR|2026-01-03 08:10:00|            NULL|Invalid sales_amount|
+--------------+----------------+--------+------------+-------------------+----------------+--------------------+



In [22]:
invalid_df = (
    invalid_df
    .withColumn(
        "error_reason",
        F.lit(
            "Invalid sales_amount"
        )
    )
)

In [24]:
(
    invalid_df
    .write
    .format("delta")
    .mode("append")
    .save(
        "../data/dlq"
    )
)

In [26]:
silver_path = (
    "../data/silver/unified_transactions_incremental"
)

target = DeltaTable.forPath(
    spark,
    silver_path
)

(
    target.alias("t")
    .merge(
        valid_df.alias("s"),
        """
        t.transaction_id
        =
        s.transaction_id
        """
    )
    .whenMatchedUpdate(
        set={
            "sales_amount":
                "s.sales_amount_num",
            "updated_at":
                "s.updated_at"
        }
    )
    .whenNotMatchedInsert(
        values={
            "transaction_id":
                "s.transaction_id",
            "transaction_date":
                "s.transaction_date",
            "store_id":
                "s.store_id",
            "sales_amount":
                "s.sales_amount_num",
            "updated_at":
                "s.updated_at"
        }
    )
    .execute()
)

In [27]:
silver = (
    spark.read
    .format("delta")
    .load(
        silver_path
    )
)

silver.orderBy(
    "transaction_id"
).show(
    truncate=False
)

+--------------+----------------+--------+------------+-------------------+
|transaction_id|transaction_date|store_id|sales_amount|updated_at         |
+--------------+----------------+--------+------------+-------------------+
|TX001         |2026-01-01      |S001    |150         |2026-01-03 08:00:00|
|TX002         |2026-01-01      |S001    |200         |2026-01-01 11:00:00|
|TX003         |2026-01-02      |S002    |300         |2026-01-02 09:00:00|
|TX004         |2026-01-03      |S003    |400         |2026-01-03 08:05:00|
|TX006         |2025-12-29      |S001    |500         |2026-01-03 08:15:00|
+--------------+----------------+--------+------------+-------------------+



In [31]:
rows_read = incremental_run2.count()

rows_valid = valid_df.count()

rows_invalid = invalid_df.count()

print(f"Rows read: {rows_read}")
print(f"Rows valid: {rows_valid}")
print(f"Rows invalid: {rows_invalid}")

Rows read: 4
Rows valid: 3
Rows invalid: 1


In [33]:
end_time = time.time()

duration_sec = round(
    end_time - start_time,
    2
)

print(duration_sec)

458.13


In [36]:
rows_inserted = 2
rows_updated = 1

metrics_pd = pd.DataFrame(
    {
        "run_id": [run_id],
        "rows_read": [rows_read],
        "rows_valid": [rows_valid],
        "rows_invalid": [rows_invalid],
        "rows_inserted": [rows_inserted],
        "rows_updated": [rows_updated],
        "duration_sec": [duration_sec]
    }
)

metrics_df = spark.createDataFrame(
    metrics_pd
)

metrics_df.show()

+---------------+---------+----------+------------+-------------+------------+------------+
|         run_id|rows_read|rows_valid|rows_invalid|rows_inserted|rows_updated|duration_sec|
+---------------+---------+----------+------------+-------------+------------+------------+
|20260621_115248|        4|         3|           1|            2|           1|      458.13|
+---------------+---------+----------+------------+-------------+------------+------------+



In [37]:
(
    metrics_df
    .write
    .mode("append")
    .parquet(
        "../data/metadata/metrics"
    )
)

In [40]:
pipeline_log_pd = pd.DataFrame(
    {
        "run_id": [run_id],
        "source_name": ["hyper"],
        "status": ["SUCCESS"],
        "start_time": [datetime.now()],
        "end_time": [datetime.now()]
    }
)

pipeline_log_df = spark.createDataFrame(
    pipeline_log_pd
)

(
    pipeline_log_df
    .write
    .mode("append")
    .parquet(
        "../data/metadata/pipeline_log"
    )
)